# Revenue Management EDA — Reusable Company Workflow (v2.1)

**Purpose:** an auditable, repeatable pricing-position study for one property, built on four PriceLabs exports (Competitor Calendar, Bookings Report, two Report Builder exports).

**Workflow:** setup → configure → run (load, validate, compute, audit) → review → charts → export evidence.

All logic lives in the tested `rm_analytics` package; this notebook only configures, displays and exports. Do not edit calculations here.

**Definitions that must not be mixed:** competitor *advertised* rates ≠ PriceLabs *market ADR* ≠ our *realized* booking ADR. A Report Builder **Stay Date Range** is not a **Booked Date** filter. Competitor unavailability is not confirmed occupancy. Nothing here proves that price caused bookings.

**Confidentiality:** never commit real exports or outputs to a public repository. `data/input/` and `output/` are git-ignored.

## 0 | Setup (Colab or local)
Finds the project folder. In Colab, if the project is not present, you will be asked to upload the project ZIP.

In [ ]:
import os, sys, zipfile, io
from pathlib import Path

def _find_root(start: Path):
    for c in [start, *start.parents, Path('/content/revenue-management-analytics')]:
        if (c / 'rm_analytics' / '__init__.py').is_file():
            return c
    content = Path('/content')
    if content.is_dir():
        for c in content.glob('*/'):
            if (c / 'rm_analytics' / '__init__.py').is_file():
                return c
    return None

PROJECT_ROOT = _find_root(Path.cwd())
if PROJECT_ROOT is None:
    try:
        from google.colab import files  # noqa: F401
    except ImportError:
        raise FileNotFoundError('Run this notebook from the project folder (the one containing rm_analytics/).')
    print('Upload the project ZIP (the whole project, not individual CSV files).')
    uploaded = files.upload()
    archive = next((n for n in uploaded if n.lower().endswith('.zip')), None)
    if archive is None:
        raise RuntimeError('No .zip file uploaded')
    dest = Path('/content/rma_upload').resolve()
    with zipfile.ZipFile(io.BytesIO(uploaded[archive])) as z:
        for m in z.namelist():
            if not (dest / m).resolve().is_relative_to(dest):
                raise ValueError(f'Unsafe path in ZIP: {m}')
        z.extractall(dest)
    PROJECT_ROOT = _find_root(dest) or next((p for p in dest.rglob('rm_analytics') if (p / '__init__.py').is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('rm_analytics/ not found inside the ZIP')
    PROJECT_ROOT = Path(PROJECT_ROOT).parent if PROJECT_ROOT.name == 'rm_analytics' else PROJECT_ROOT

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import rm_analytics
from rm_analytics import AnalysisConfig, run_analysis, export_run
from rm_analytics.charts import build_figures
pd.set_option('display.max_colwidth', 160)
print('Project:', PROJECT_ROOT, '| rm_analytics', rm_analytics.__version__, '| pandas', pd.__version__)

## 1 | Configure this run

* `property_name`: exact listing name as in the exports (an exact match is preferred; a partial match must be unique).
* `booking_start/end`: **Booked Date** window (reservation creation). Inclusive, any time of day.
* `historical_stay_*` and `forward_*`: the **Stay Date Ranges you selected in Report Builder**. They are cross-checked against the Bookings report; mismatches are reported.
* `data_dir`: `data/input` for real work. To try the tool, run the next cell's optional line to create synthetic example data and use `data/example_synthetic`.
* Attestations are recorded in the manifest; set them to `True` only after doing the check.

In [ ]:
# Optional: create the synthetic example dataset (no real data) and try the tool with it.
# from rm_analytics.synthetic import generate_synthetic_dataset; generate_synthetic_dataset('data/example_synthetic')

DATA_DIR = os.environ.get('RMA_DATA_DIR', 'data/input')
IS_SYNTHETIC = 'synthetic' in DATA_DIR

cfg = AnalysisConfig(
    property_name=os.environ.get('RMA_PROPERTY', 'YOUR LISTING NAME'),
    booking_start='2026-10-11',
    booking_end='2026-11-02',
    historical_stay_start='2026-10-11',
    historical_stay_end='2026-11-02',
    forward_start='2026-11-03',
    forward_end='2027-01-31',
    data_dir=DATA_DIR,
    min_active_comps=5,             # minimum available comps for a date to be compared
    calendar_date_format='auto',    # or e.g. '%d/%m/%Y' if detection reports ambiguity
    bookings_date_format='auto',
    duplicate_policy='warn',        # 'drop' removes the second listing of a near-identical pair
    exclude_comps=(),               # comp names to exclude after review
    analyst='',                     # your name
    reviewer='',
    attest_report_periods_verified=False,
    attest_comp_set_reviewed=False,
    attest_storage_approved=False,
    is_synthetic_example=IS_SYNTHETIC,
)
print('Input files:'); [print(' ', k, '->', p) for k, p in cfg.input_paths().items()]

## 2 | Run: load, validate, compute, audit
Structural problems (unparseable money/dates, unknown booking statuses, ambiguous listing names, calendar gaps) stop the run with a clear message. Fix the input; do not bypass the check.

In [ ]:
result = run_analysis(cfg)
audit_df = result.audit.to_frame()
print('Subject listing:', result.calendar.target)
print('Comps used:', len(result.comps), '| dropped as duplicates:', result.dropped_comps)
print('Audit:', result.audit.counts())
display(audit_df[audit_df.status != 'PASS'])
print('DECISION STATUS:', result.audit.decision_status)
if result.audit.decision_status != 'VALIDATED':
    print('REVIEW REQUIRED: inspect audit warnings and document resolution before relying on conclusions.')


## 3 | Executive metrics
Pricing metrics compare **advertised** rates on our **available** dates in the forward window. The confidence interval accounts for within-week correlation. Compare the bias-sensitivity row with the main gap: a large difference means the availability filter is driving the result.

In [ ]:
display(result.summary_table.round(2))

## 4 | ADR, occupancy and RevPAR versus PriceLabs market
RevPAR proxy = ADR × occupancy. It answers whether a lower rate was compensated by higher occupancy. Occupancy definitions may differ between Adjusted and Market occupancy; verify in PriceLabs.

In [ ]:
display(result.market.T)
print('Reconciliation of bookings (allocated to stay dates) vs Report Builder:')
display(result.reconciliation)

## 5 | Daily detail

In [ ]:
display(result.band_by_day_group)
cols = ['our_price','n_comps','comp_p25','comp_median','comp_p75','comp_std','gap_vs_median_pct','gap_vs_all_priced_median_pct','our_min_stay','comp_median_min_stay','price_band','exclusion_reason']
display(result.positions[cols].round(2).head(20))
if len(result.duplicates):
    print('Probable duplicate homes:'); display(result.duplicates)
print('Minimum-stay sensitivity:')
from rm_analytics.export import _minimum_stay_sensitivity
display(_minimum_stay_sensitivity(result).round(2))


## 6 | Charts

In [ ]:
figures = build_figures(result)
plt.show()

## 7 | Export evidence
Creates `output/<run_id>/` with tables, minimum-stay sensitivity, `REVIEW_SUMMARY.md`, charts, audit and `run_manifest.json` (input and output SHA-256, config, code and environment versions). Runs are **VALIDATED** (zero audit warnings/failures), **REVIEW_REQUIRED** (one or more WARN), or **FAILED** (one or more FAIL). Review-required runs are exported to folders ending in `_REVIEW_REQUIRED` and may not be treated as validated. All price changes require separate approval. In Colab, download the folder before the session ends.

In [ ]:
run_dir = export_run(result, figures)
print('Exported to', run_dir)
for p in sorted(run_dir.iterdir()):
    print(' -', p.name)
if result.audit.has_failures:
    raise RuntimeError(f'Audit FAIL checks present; run exported as FAILED for traceability: {run_dir.name}')
print('DECISION STATUS:', result.audit.decision_status)
if result.audit.decision_status == 'REVIEW_REQUIRED':
    print('Not yet validated. Read REVIEW_SUMMARY.md and resolve or document review items.')


## 8 | Reviewer checklist and interpretation

1. Confirm property identity, date windows and that all four exports come from the same PriceLabs filters.
2. Resolve WARNs, or document your acceptance and rationale in the review record; the generated status remains REVIEW_REQUIRED until rerun with all checks passing. Pay attention to reconciliation, minimum-stay comparability and bias sensitivity.
3. Spot-check five dates against the PriceLabs UI, including a weekend and a low-comp date.
4. Interpret historical and forward market comparisons separately. Check RevPAR, not only ADR or occupancy.
5. Consider alternative explanations: listing age and review count, launch discounts, search ranking, minimum stays, cleaning fees, seasonality.
6. Any rate change is a **pricing test** with a hypothesis, dates, magnitude, monitoring metric and rollback rule, approved separately.

**Conclusion framework:** Observation → Evidence → Alternative explanations → Proposed test → Risk/monitoring → Approval status.